# IOAI — 2026 Round2 Central Force (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test_field.npz'):
    os.system('wget -q https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-round2-central-force/test_field.npz -O data/test_field.npz')
# 학습셋(300MB)은 노트북 첫 셀이 HuggingFace 에서 직접 받는다.
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Central Force — USAAIO 2026 Round 2 (Day 1, Problem 2, open-ended)

2D 벡터장 관측 `(64,64,2)` 에서 **단일 소스 위치 `(x_s,y_s)`** 와 소스로부터의 거리별 **크기함수 `f(r)`**(100개
반지름 `r=0.02,0.04,…,2.00`)를 복원하는 **역문제**다. baseline 은 벡터장을 이미지처럼 보고 CNN 으로 102개 값
(위치 2 + 크기 100)을 회귀한다.

학습셋(10000, 라벨 포함)은 HuggingFace 에서 자동 다운로드하고, 예측 대상은 `data/test_field.npz`(field 만)이다.
`submission.csv`(`id, source_x, source_y, f_0..f_99`)를 만든다.

**채점**: 표본별 `error = 1e3·(|Δx|+|Δy|) + 3e-3·RMSE(f)` 의 평균(**낮을수록 좋음**). GPU 권장(수 초).

In [ ]:
import os, urllib.request, numpy as np
os.makedirs("data", exist_ok=True)
# 학습셋(10000, 라벨 포함)은 HuggingFace 공개 저장소에서 받는다(300MB, 최초 1회).
BASE="https://huggingface.co/datasets/usaaio-official/2026_USAAIO_Round2/resolve/main/Central_force_single_source"
if not os.path.exists("data/train.npz"):
    print("downloading train.npz (~300MB) ..."); urllib.request.urlretrieve(f"{BASE}/train.npz", "data/train.npz")
tr = np.load("data/train.npz")
te = np.load("data/test_field.npz")          # 예측 대상(field 만, 라벨 숨김) — 저장소에 포함
print("train", tr["field"].shape, "| test_field", te["field"].shape)

In [ ]:
import torch, torch.nn as nn
dev = "cuda" if torch.cuda.is_available() else "cpu"

Xtr = torch.tensor(tr["field"]).permute(0,3,1,2).contiguous()      # (N,2,64,64)
ytr = torch.tensor(np.column_stack([tr["source_x"], tr["source_y"], tr["f_at_r"]]), dtype=torch.float32)
fmean, fstd = ytr[:,2:].mean(0), ytr[:,2:].std(0)+1e-6             # f(r) 정규화(스케일 큼)
ytr_n = ytr.clone(); ytr_n[:,2:] = (ytr[:,2:]-fmean)/fstd

class FieldCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(2,32,3,2,1), nn.ReLU(), nn.Conv2d(32,64,3,2,1), nn.ReLU(),
                               nn.Conv2d(64,128,3,2,1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Sequential(nn.Linear(128,256), nn.ReLU(), nn.Linear(256,102))
    def forward(self, x): return self.h(self.c(x))

In [ ]:
m = FieldCNN().to(dev); opt = torch.optim.Adam(m.parameters(), 1e-3); lossf = nn.MSELoss()
Xg, yg = Xtr.to(dev), ytr_n.to(dev); fm, fs = fmean.to(dev), fstd.to(dev)
bs, N = 256, len(Xg)
for ep in range(5):
    perm = torch.randperm(N, device=dev)
    for i in range(0, N, bs):
        idx = perm[i:i+bs]; opt.zero_grad()
        loss = lossf(m(Xg[idx]), yg[idx]); loss.backward(); opt.step()
print("final loss", round(loss.item(), 4))

In [ ]:
import pandas as pd
Xte = torch.tensor(te["field"]).permute(0,3,1,2).contiguous().to(dev)
m.eval()
with torch.no_grad():
    out = m(Xte); out[:,2:] = out[:,2:]*fs + fm; out = out.cpu().numpy()
cols = ["source_x","source_y"] + [f"f_{i}" for i in range(100)]
sub = pd.DataFrame(out, columns=cols); sub.insert(0, "id", [str(x) for x in te["id"]])
sub.to_csv("submission.csv", index=False)
print("saved submission.csv", sub.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)